# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [7]:
"Rule 1: Staleness. The idea behind FlyRank's refresh flags is if it's been sitting untouched a long time, it's probably losing steam. I group pages into buckets by how long since their last update, then check the decline rate in each bucket. If older buckets consistently decline more, that confirms the assumption but if it's inconsistent then the assumption doesn't hold cleanly."

"Rule 2: CTR vs. position. Pages ranked 1-3 should get way more clicks than pages ranked 50, purely because people scroll from the top. I bucket the pages by position and check average CTR per bucket. If it drops smoothly as position gets worse, that's a clean, reliable signal."

#SETUP
%pip -q install duckdb huggingface_hub
import duckdb
import pandas as pd
from google.colab import userdata
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

DEV_MONTH_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
DIM_CONTENT = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"
DIM_CLIENTS = "hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet"

#Initializing Labels for Signals
labels = con.sql(f"""
    SELECT content_hash_id,
      AVG(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_clicks END) AS clicks_first_half,
      AVG(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_clicks END) AS clicks_second_half
    FROM read_parquet('{DEV_MONTH_PATH}')
    GROUP BY content_hash_id
""").df()
labels['is_declining'] = (labels['clicks_second_half'] < labels['clicks_first_half']).astype(int)

#SIGNAL CHECK
content_agg = con.sql(f"""
    SELECT
      f.content_hash_id,
      SUM(f.gsc_clicks) AS clicks_total,
      SUM(f.gsc_impressions) AS impressions_total,
      AVG(f.gsc_avg_position) AS avg_position,
      MAX(f.report_date) AS last_report_date
    FROM read_parquet('{DEV_MONTH_PATH}') f
    GROUP BY f.content_hash_id
""").df()

# reattach the is_declining label and dim_content's content_updated_date
staleness = con.sql(f"""
    SELECT content_hash_id, content_updated_date
    FROM read_parquet('{DIM_CONTENT}')
""").df()

df = content_agg.merge(staleness, on='content_hash_id').merge(
    labels[['content_hash_id','is_declining']], on='content_hash_id'
).dropna(subset=['content_updated_date'])

df['days_since_update'] = (pd.to_datetime(df['last_report_date']) - pd.to_datetime(df['content_updated_date'])).dt.days
df['ctr'] = df['clicks_total'] / df['impressions_total'].replace(0, pd.NA)

#SIGNAL 1
print("Signal 1")
df['staleness_bucket'] = pd.cut(df['days_since_update'],
    bins=[-1, 30, 90, 180, 365, 999999],
    labels=['0-30d','31-90d','91-180d','181-365d','365d+'])

staleness_table = df.groupby('staleness_bucket', observed=True).agg(
    n=('content_hash_id','count'),
    decline_rate=('is_declining','mean')
).round(3)
print(staleness_table)
print("Total n:", df.shape[0])
print()

#SIGNAL 2
print("Signal 2")
df['position_bucket'] = pd.cut(df['avg_position'],
    bins=[0,3,10,20,50,999], labels=['1-3','4-10','11-20','21-50','50+'])

ctr_table = df.groupby('position_bucket', observed=True).agg(
    n=('content_hash_id','count'),
    avg_ctr=('ctr','mean')
).round(4)
print(ctr_table)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 1
                      n  decline_rate
staleness_bucket                     
0-30d               975         0.032
31-90d            29680         0.211
91-180d            3608         0.026
181-365d           3816         0.001
Total n: 331437

Signal 2
                     n   avg_ctr
position_bucket                 
1-3              16144  0.010589
4-10             81988  0.004926
11-20            32203  0.003211
21-50            33288  0.002287
50+              11681  0.000903


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [11]:
import os

def score_row(row):
    reason = None
    score = 0
    if row['staleness_bucket'] in ['91-180d'] and row['is_declining']:
        score += 2
        reason = "STALE_AND_DECLINING"
    elif row['position_bucket'] in ['4-10','11-20'] and row['ctr'] < 0.01:
        score += 2
        reason = "LOW_CTR_FOR_POSITION"
    elif row['is_declining']:
        score += 1
        reason = "DECLINING_ONLY"
    else:
        reason = "NO_FLAG"

    if score >= 2:
        action = "PRIORITY_REVIEW"
    elif score == 1:
        action = "WATCH"
    else:
        action = "NO_ACTION"
    return pd.Series({'score': score, 'reason_code': reason, 'action': action})

df[['score','reason_code','action']] = df.apply(score_row, axis=1)

ranked = df[df['impressions_total'] >= 50].sort_values(
    ['score','impressions_total'], ascending=[False, False]
)[['content_hash_id','score','reason_code','action','staleness_bucket','position_bucket','ctr','is_declining','impressions_total']]

os.makedirs('work/outputs', exist_ok=True)
ranked.to_csv('work/outputs/baseline_action_score.csv', index=False)
print("Wrote", len(ranked), "rows")
ranked.head(20)

Wrote 116114 rows


,content_hash_id,score,reason_code,action,staleness_bucket,position_bucket,ctr,is_declining,impressions_total
58741,content_e8a52cf3d5988c07,2,LOW_CTR_FOR_POSITION,PRIORITY_REVIEW,NaN,11-20,0.002731,1,244931.0
45375,content_44f34c0a90047651,2,LOW_CTR_FOR_POSITION,PRIORITY_REVIEW,NaN,4-10,0.000113,0,212404.0
225122,content_7172a7fad43f0998,2,LOW_CTR_FOR_POSITION,PRIORITY_REVIEW,NaN,4-10,0.004187,1,205867.0
60254,content_f107e54b10b43725,2,LOW_CTR_FOR_POSITION,PRIORITY_REVIEW,NaN,4-10,0.005082,0,195997.0
59641,content_b99ea6861864dea5,2,LOW_CTR_FOR_POSITION,PRIORITY_REVIEW,NaN,4-10,0.001858,1,194337.0
225011,content_acbcc847f8996314,2,LOW_CTR_FOR_POSITION,PRIORITY_REVIEW,NaN,4-10,0.001534,1,170808.0
318403,content_471d9cabce329a66,2,LOW_CTR_FOR_POSITION,PRIORITY_REVIEW,NaN,4-10,0.002402,1,164885.0
671,content_fd2117c2c6790e4b,2,LOW_CTR_FOR_POSITION,PRIORITY_REVIEW,NaN,4-10,0.002699,1,151166.0
183904,content_34a70fea29d15f24,2,LOW_CTR_FOR_POSITION,PRIORITY_REVIEW,NaN,4-10,0.000301,0,143019.0
166590,content_e241d6415ac9e534,2,LOW_CTR_FOR_POSITION,PRIORITY_REVIEW,NaN,4-10,0.00241,0,142304.0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [12]:
"1.e8a52cf3d5988c07 — PRIORITY_REVIEW, LOW_CTR_FOR_POSITION. CTR 0.27% at position 11–20, already declining. Wrong if 244,931 impressions are mostly bot/scraper traffic inflating the denominator rather than real search users"
"2.44f34c0a90047651 — PRIORITY_REVIEW. CTR 0.01% at position 4–10 — extremely low for that tier. Wrong if the ranking keyword is mismatched to page intent (people see it, correctly skip it) rather than a fixable CTR problem."
"3.7172a7fad43f0998 — PRIORITY_REVIEW. CTR 0.42%, close to bucket average — borderline flag. Wrong if this is simply normal variance, not a real problem; shouldn't be top-priority."
"4.f107e54b10b43725 — PRIORITY_REVIEW. CTR 0.51%, near-average for the bucket, not declining. Likely a weak pick — flagged mostly by the loose threshold, not a real anomaly."
"5.b99ea6861864dea5 — PRIORITY_REVIEW. CTR 0.19%, declining. Wrong if the drop is seasonal/topic-driven rather than a title/snippet problem."
"6.acbcc847f8996314 — PRIORITY_REVIEW. CTR 0.15%, declining, high volume (170k impressions). Solid candidate — wrong only if impression count itself is inflated by duplicate tracking."
"7.471d9cabce329a66 — PRIORITY_REVIEW. CTR 0.24%, declining. Wrong if page recently changed target keyword, making position 4–10 not comparable across the month."
"8.fd2117c2c6790e4b — PRIORITY_REVIEW. CTR 0.27%, declining. Same caveat as above — wrong if underlying keyword/query shifted mid-month."
"9.34a70fea29d15f24 — PRIORITY_REVIEW. CTR 0.03%, not declining despite very low CTR. Worth investigating why it's stable despite poor CTR — could mean traffic comes from elsewhere (social, direct) and GSC CTR isn't the full picture."
"10.e241d6415ac9e534 — PRIORITY_REVIEW. CTR 0.24%, not declining, near-average-ish. Weak pick — not clearly worse than typical for its bucket."
"11.f43118e089ecc69a — PRIORITY_REVIEW. Only row with a valid staleness bucket (31–90d) — declining and low CTR together. Strongest, most defensible pick in the batch; wrong only if the topic itself is seasonal."
"12.66288edeb93b7c4f — PRIORITY_REVIEW. CTR 0.57% at position 11–20, not declining. Weak pick — CTR is actually reasonable for that worse position tier."
"13.f352b7cfd0b2f434 — PRIORITY_REVIEW. CTR 0.21%, declining. Reasonable candidate; wrong if declining due to a competitor's new content rather than this page's own quality."
"14.8e1334d6356668e3 — PRIORITY_REVIEW. CTR 0.0007% (essentially zero clicks on 135k impressions) — the most extreme anomaly in the batch. Wrong only if this is a tracking/parsing bug rather than a real page."
"15.7c6373141eae744a — PRIORITY_REVIEW. CTR 0.06%, declining. Solid candidate; wrong if snippet/meta is fine and the real issue is ranking volatility not reflected in the monthly average."
"16.00d4fdf6e48a2d38 — PRIORITY_REVIEW. CTR 0.37%, not declining. Weak pick — reasonably close to expected range."
"17.fe3fd3422852d721 — PRIORITY_REVIEW. CTR 0.30%, not declining. Weak pick, same reasoning."
"18.fec55986a1868d62 — PRIORITY_REVIEW. CTR 0.0008%, declining — near-zero clicks like #14. Strong candidate for a real (not noise) problem given the scale of impressions."
"19.95ff62babbfac9c7 — PRIORITY_REVIEW. CTR 0.19%, not declining. Borderline — worth a second look but not top priority."
"20.3b6e4c8d9a0a5c9c — PRIORITY_REVIEW. CTR 0.32%, declining. Reasonable candidate, same generic CTR-fix caveat as others abo"

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [12]:
"I pick 4, 10, 12, 16, 17, 19 explicitly since these sit close to the bucket's normal CTR range and only got flagged because the < 0.01 threshold is looser than the actual average"

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.